# 📝 Virtual Environments, pip, Poetry, uv
### Exercises & Solutions — 22 Problems

This notebook is exercises-and-solutions only. It assumes you've already covered the
concept notebook for this topic. Each problem targets a **distinct function, pattern,
or real-world scenario** so that working through all of them gives you practical
exposure to everything commonly used on the job.

**Coverage map:**

- Environment inspection & venv mechanics (1-4)
- pip workflows: install, freeze, constraints (5-9)
- Dependency resolution & conflict detection (10-13)
- requirements.txt / lock file generation & parsing (14-17)
- Poetry & uv command construction and pyproject.toml handling (18-22)


---


### 1. Detect if Running Inside a Virtual Environment

Write `is_in_venv() -> bool` that correctly detects virtual environment activation across both `venv` and `virtualenv` conventions.

In [ ]:
import sys

def is_in_venv() -> bool:
    return (hasattr(sys, "real_prefix") or                      # virtualenv (older)
            (hasattr(sys, "base_prefix") and sys.base_prefix != sys.prefix))  # venv (stdlib)

print("In virtual environment:", is_in_venv())
print("sys.prefix:", sys.prefix)
print("sys.base_prefix:", getattr(sys, "base_prefix", "N/A"))

### 2. List Installed Packages with Versions

Write a function listing all installed packages and their versions using `importlib.metadata`, sorted alphabetically.

In [ ]:
import importlib.metadata as metadata

def list_installed_packages():
    return sorted(
        [(d.metadata["Name"], d.version) for d in metadata.distributions()],
        key=lambda x: x[0].lower()
    )

packages = list_installed_packages()
print(f"Total installed: {len(packages)}")
for name, version in packages[:8]:
    print(f"  {name}=={version}")

### 3. Find a Specific Package's Dependencies

Write `get_dependencies(package_name)` that uses `importlib.metadata.requires()` to list a package's declared dependencies.

In [ ]:
import importlib.metadata as metadata

def get_dependencies(package_name):
    try:
        return metadata.requires(package_name) or []
    except metadata.PackageNotFoundError:
        return None

deps = get_dependencies("pip")
print(f"pip's declared dependencies: {deps}")

deps2 = get_dependencies("nonexistent-package-xyz")
print(f"Nonexistent package result: {deps2}")

### 4. Check Python Version Compatibility

Write `check_python_compatibility(min_version, max_version)` parsing version tuples and comparing against the running interpreter.

In [ ]:
import sys

def check_python_compatibility(min_version, max_version=None):
    current = sys.version_info[:2]
    if current < min_version:
        return False, f"Python {current} is below minimum {min_version}"
    if max_version and current >= max_version:
        return False, f"Python {current} is at/above max {max_version}"
    return True, f"Python {current} is compatible"

print(check_python_compatibility((3, 8)))
print(check_python_compatibility((3, 8), (3, 10)))
print(check_python_compatibility((4, 0)))

### 5. Parse pip freeze Output into a Dict

Write `parse_freeze_output(text) -> dict` that parses `pip freeze`-style text (`package==version` per line) into a dict, handling edge cases (comments, blank lines, git URLs).

In [ ]:
def parse_freeze_output(text: str) -> dict:
    result = {}
    for line in text.strip().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("-e ") or "@" in line:    # editable installs / git URLs
            continue
        if "==" in line:
            name, version = line.split("==", 1)
            result[name.lower()] = version
    return result

sample = """
# This is a comment
requests==2.31.0
pandas==2.1.0

numpy==1.26.0
-e git+https://github.com/user/repo.git#egg=mypackage
"""
print(parse_freeze_output(sample))

### 6. Generate requirements.txt from a Dict

Write `dict_to_requirements(deps: dict, dev_deps: dict = None) -> str` generating well-formatted, sorted requirements.txt content with section comments.

In [ ]:
def dict_to_requirements(deps: dict, dev_deps: dict = None) -> str:
    lines = ["# Production dependencies"]
    for name in sorted(deps):
        lines.append(f"{name}=={deps[name]}")
    if dev_deps:
        lines.append("")
        lines.append("# Development dependencies")
        for name in sorted(dev_deps):
            lines.append(f"{name}=={dev_deps[name]}")
    return "\n".join(lines)

content = dict_to_requirements(
    {"requests": "2.31.0", "fastapi": "0.104.0"},
    {"pytest": "7.4.0", "black": "23.9.0"}
)
print(content)

### 7. Version Constraint Satisfaction Checker

Write `satisfies_constraint(version: str, constraint: str) -> bool` supporting `>=`, `<=`, `==`, `!=` against simple X.Y.Z version strings (no need for full PEP 440).

In [ ]:
def parse_version(v):
    return tuple(int(x) for x in v.split("."))

def satisfies_constraint(version: str, constraint: str) -> bool:
    for op in [">=", "<=", "==", "!=", ">", "<"]:
        if constraint.startswith(op):
            target = constraint[len(op):].strip()
            v, t = parse_version(version), parse_version(target)
            if op == ">=": return v >= t
            if op == "<=": return v <= t
            if op == "==": return v == t
            if op == "!=": return v != t
            if op == ">":  return v > t
            if op == "<":  return v < t
    raise ValueError(f"Unrecognized constraint: {constraint}")

print(satisfies_constraint("2.31.0", ">=2.28.0"))
print(satisfies_constraint("2.20.0", ">=2.28.0"))
print(satisfies_constraint("1.0.0", "!=1.0.0"))

### 8. Build a pip install Command from Constraints

Write `build_pip_install_cmd(packages: dict, upgrade=False, no_cache=False) -> str` constructing a valid pip install command line from a constraints dict.

In [ ]:
def build_pip_install_cmd(packages: dict, upgrade=False, no_cache=False) -> str:
    parts = ["pip", "install"]
    if upgrade:
        parts.append("--upgrade")
    if no_cache:
        parts.append("--no-cache-dir")
    for name, constraint in packages.items():
        parts.append(f'"{name}{constraint}"' if constraint else name)
    return " ".join(parts)

cmd = build_pip_install_cmd(
    {"requests": ">=2.28,<3.0", "pandas": "==2.1.0", "click": ""},
    upgrade=True
)
print(cmd)

### 9. Simulate --dry-run Dependency Preview

Write a function that, given a desired package dict and an already-installed dict, classifies each desired package as `to_install`, `to_upgrade`, `to_downgrade`, or `already_satisfied`.

In [ ]:
def plan_install(desired: dict, installed: dict) -> dict:
    plan = {"to_install": [], "to_upgrade": [], "to_downgrade": [], "already_satisfied": []}
    for name, target_version in desired.items():
        if name not in installed:
            plan["to_install"].append(name)
        else:
            current = tuple(map(int, installed[name].split(".")))
            target = tuple(map(int, target_version.split(".")))
            if current < target:
                plan["to_upgrade"].append(f"{name}: {installed[name]} -> {target_version}")
            elif current > target:
                plan["to_downgrade"].append(f"{name}: {installed[name]} -> {target_version}")
            else:
                plan["already_satisfied"].append(name)
    return plan

desired = {"requests": "2.31.0", "pandas": "2.0.0", "numpy": "1.20.0", "click": "8.1.0"}
installed = {"requests": "2.28.0", "pandas": "2.1.0", "numpy": "1.20.0"}
import json
print(json.dumps(plan_install(desired, installed), indent=2))

### 10. Detect Direct Version Conflicts Between Two Projects

Write `find_conflicts(proj_a: dict, proj_b: dict) -> list` identifying packages required by both projects with INCOMPATIBLE exact-pinned versions.

In [ ]:
def find_conflicts(proj_a: dict, proj_b: dict) -> list:
    conflicts = []
    shared = set(proj_a) & set(proj_b)
    for pkg in shared:
        if proj_a[pkg] != proj_b[pkg]:
            conflicts.append({"package": pkg, "project_a": proj_a[pkg], "project_b": proj_b[pkg]})
    return conflicts

a = {"requests": "2.28.0", "django": "4.2", "celery": "5.3.0"}
b = {"requests": "2.31.0", "fastapi": "0.104.0", "celery": "5.3.0"}
print(find_conflicts(a, b))

### 11. Transitive Dependency Graph Construction

Build a simple dependency graph from a dict of `{package: [direct_deps]}` and write a function returning ALL transitive dependencies for a given package (handling cycles safely).

In [ ]:
def transitive_deps(graph: dict, package: str, seen=None) -> set:
    if seen is None:
        seen = set()
    if package in seen:
        return seen           # cycle protection
    seen.add(package)
    for dep in graph.get(package, []):
        transitive_deps(graph, dep, seen)
    return seen

graph = {
    "webapp": ["flask", "requests"],
    "flask": ["werkzeug", "jinja2"],
    "requests": ["urllib3", "certifi"],
    "werkzeug": [],
    "jinja2": ["markupsafe"],
    "urllib3": [],
    "certifi": [],
    "markupsafe": [],
}
result = transitive_deps(graph, "webapp")
print(sorted(result - {"webapp"}))

### 12. Detect Circular Dependencies

Write `has_circular_dependency(graph: dict) -> bool` using DFS with a recursion-stack to detect cycles in a dependency graph (which would indicate a packaging error).

In [ ]:
def has_circular_dependency(graph: dict) -> bool:
    visited, in_stack = set(), set()

    def dfs(node):
        visited.add(node)
        in_stack.add(node)
        for neighbor in graph.get(node, []):
            if neighbor in in_stack:
                return True               # back edge = cycle
            if neighbor not in visited and dfs(neighbor):
                return True
        in_stack.remove(node)
        return False

    return any(dfs(node) for node in graph if node not in visited)

clean_graph = {"a": ["b"], "b": ["c"], "c": []}
cyclic_graph = {"a": ["b"], "b": ["c"], "c": ["a"]}   # a -> b -> c -> a !

print("Clean graph has cycle:", has_circular_dependency(clean_graph))
print("Cyclic graph has cycle:", has_circular_dependency(cyclic_graph))

### 13. Topological Sort for Install Order

Write `install_order(graph: dict) -> list` performing a topological sort so dependencies are always listed BEFORE the packages that need them.

In [ ]:
def install_order(graph: dict) -> list:
    visited, order = set(), []
    def dfs(node):
        if node in visited:
            return
        visited.add(node)
        for dep in graph.get(node, []):
            dfs(dep)
        order.append(node)        # add AFTER all dependencies are processed

    for node in graph:
        dfs(node)
    return order

graph = {
    "webapp": ["flask", "requests"],
    "flask": ["werkzeug"],
    "requests": ["urllib3"],
    "werkzeug": [], "urllib3": [],
}
print(install_order(graph))
print("Note: dependencies always appear BEFORE the packages needing them")

### 14. Parse a Poetry-style pyproject.toml Dependency Section

Write a parser extracting dependencies from a Poetry-style `[tool.poetry.dependencies]` TOML-like dict structure into a normalized `{name: constraint}` dict.

In [ ]:
def parse_poetry_deps(pyproject_dict: dict) -> dict:
    deps = pyproject_dict.get("tool", {}).get("poetry", {}).get("dependencies", {})
    normalized = {}
    for name, spec in deps.items():
        if name == "python":
            continue          # python version constraint, not a real package
        if isinstance(spec, dict):
            normalized[name] = spec.get("version", "*")
        else:
            normalized[name] = spec
    return normalized

sample = {
    "tool": {"poetry": {"dependencies": {
        "python": "^3.10",
        "requests": "^2.31.0",
        "fastapi": {"version": ">=0.100.0", "extras": ["all"]},
    }}}
}
print(parse_poetry_deps(sample))

### 15. Convert Poetry Caret/Tilde Constraints to pip-style

Write `poetry_constraint_to_pip(spec: str) -> str` converting Poetry's `^1.2.3` (compatible release) and `~1.2.3` (minor-level) syntax into pip-compatible range constraints.

In [ ]:
def poetry_constraint_to_pip(spec: str) -> str:
    if spec.startswith("^"):
        version = spec[1:]
        major = version.split(".")[0]
        next_major = int(major) + 1
        return f">={version},<{next_major}.0.0"
    elif spec.startswith("~"):
        parts = version_parts = spec[1:].split(".")
        if len(parts) >= 2:
            next_minor = f"{parts[0]}.{int(parts[1])+1}.0"
            return f">={spec[1:]},<{next_minor}"
        return f">={spec[1:]}"
    return spec   # already pip-style or exact

print(poetry_constraint_to_pip("^2.31.0"))   # caret: compatible within major version
print(poetry_constraint_to_pip("~1.4.2"))     # tilde: compatible within minor version
print(poetry_constraint_to_pip(">=1.0,<2.0")) # already pip-style, unchanged

### 16. Generate a Minimal uv-compatible pyproject.toml

Write `generate_pyproject(name, version, deps: dict, dev_deps: dict = None) -> str` producing valid PEP 621 `[project]` TOML content compatible with `uv`.

In [ ]:
def generate_pyproject(name, version, deps: dict, dev_deps: dict = None) -> str:
    dep_lines = ",\n".join(f'    "{pkg}{constraint}"' for pkg, constraint in deps.items())
    content = f'''[project]
name = "{name}"
version = "{version}"
requires-python = ">=3.9"
dependencies = [
{dep_lines}
]
'''
    if dev_deps:
        dev_lines = ",\n".join(f'    "{pkg}{constraint}"' for pkg, constraint in dev_deps.items())
        content += f'''
[dependency-groups]
dev = [
{dev_lines}
]
'''
    return content

print(generate_pyproject(
    "my-service", "0.1.0",
    {"fastapi": ">=0.104.0", "requests": ">=2.31.0"},
    {"pytest": ">=7.4.0"}
))

### 17. Diff Two Lock Files for Drift Detection

Write `diff_lockfiles(old: dict, new: dict) -> dict` comparing two `{package: version}` lock snapshots, categorizing changes as added/removed/upgraded/downgraded.

In [ ]:
def diff_lockfiles(old: dict, new: dict) -> dict:
    changes = {"added": [], "removed": [], "upgraded": [], "downgraded": []}
    for pkg in set(new) - set(old):
        changes["added"].append(f"{pkg}=={new[pkg]}")
    for pkg in set(old) - set(new):
        changes["removed"].append(f"{pkg}=={old[pkg]}")
    for pkg in set(old) & set(new):
        if old[pkg] != new[pkg]:
            old_v = tuple(map(int, old[pkg].split(".")))
            new_v = tuple(map(int, new[pkg].split(".")))
            direction = "upgraded" if new_v > old_v else "downgraded"
            changes[direction].append(f"{pkg}: {old[pkg]} -> {new[pkg]}")
    return changes

old_lock = {"requests": "2.28.0", "numpy": "1.24.0", "old-pkg": "1.0.0"}
new_lock = {"requests": "2.31.0", "numpy": "1.20.0", "new-pkg": "3.0.0"}
import json
print(json.dumps(diff_lockfiles(old_lock, new_lock), indent=2))

### 18. Build uv add Commands from a Requirements Dict

Write `build_uv_add_commands(deps: dict, dev: bool=False) -> list[str]` generating the equivalent `uv add` shell commands for a set of dependencies.

In [ ]:
def build_uv_add_commands(deps: dict, dev: bool = False) -> list:
    flag = " --dev" if dev else ""
    return [f"uv add \"{name}{constraint}\"{flag}" for name, constraint in deps.items()]

prod_cmds = build_uv_add_commands({"fastapi": ">=0.104.0", "uvicorn": ">=0.24.0"})
dev_cmds = build_uv_add_commands({"pytest": ">=7.4.0", "ruff": ">=0.1.0"}, dev=True)
for cmd in prod_cmds + dev_cmds:
    print(cmd)

### 19. Migration Script Generator: requirements.txt -> Poetry add commands

Write `requirements_to_poetry_commands(requirements_text: str) -> list[str]` that parses simple requirements.txt content and outputs `poetry add` commands.

In [ ]:
def requirements_to_poetry_commands(requirements_text: str) -> list:
    commands = []
    for line in requirements_text.strip().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        commands.append(f"poetry add \"{line}\"")
    return commands

reqs = """
requests>=2.28,<3.0
pandas==2.1.0
# this is a dev tool, handle separately
fastapi>=0.100.0
"""
for cmd in requirements_to_poetry_commands(reqs):
    print(cmd)

### 20. Dependency Health Scanner

Write `scan_dependency_health(deps: dict) -> dict` flagging packages that are unpinned, pre-1.0 (potentially unstable), or use overly broad wildcard constraints.

In [ ]:
import re

def scan_dependency_health(deps: dict) -> dict:
    report = {"unpinned": [], "prerelease": [], "wildcard": [], "healthy": []}
    for name, constraint in deps.items():
        if constraint in ("", "*"):
            report["wildcard"].append(name)
        elif not constraint:
            report["unpinned"].append(name)
        else:
            version_match = re.search(r"(\d+)\.", constraint)
            if version_match and version_match.group(1) == "0":
                report["prerelease"].append(f"{name}{constraint}")
            else:
                report["healthy"].append(f"{name}{constraint}")
    return report

deps = {
    "requests": ">=2.28,<3.0",
    "experimental-lib": ">=0.4.0",
    "internal-tool": "*",
    "numpy": ">=1.24",
}
import json
print(json.dumps(scan_dependency_health(deps), indent=2))

### 21. Environment Variable-Based Package Source Selector

Write a function choosing between a private package index and PyPI based on an environment variable, building the correct `pip install --index-url` command.

In [ ]:
import os

def build_install_command(package: str, private_index_var="PRIVATE_PYPI_URL") -> str:
    private_url = os.environ.get(private_index_var)
    if private_url:
        return f"pip install --index-url {private_url} --extra-index-url https://pypi.org/simple {package}"
    return f"pip install {package}"

print("Without private index:", build_install_command("mypackage"))

os.environ["PRIVATE_PYPI_URL"] = "https://pypi.company.com/simple"
print("With private index:", build_install_command("mypackage"))

### 22. Full Tool Recommendation Engine

Write `recommend_tool(project_type: str, team_size: int, needs_publishing: bool) -> str` encoding the decision logic for venv/pip vs Poetry vs uv based on project characteristics.

In [ ]:
def recommend_tool(project_type: str, team_size: int, needs_publishing: bool) -> str:
    if project_type == "script" and team_size == 1:
        return "venv + pip (simplest option, no team coordination needed)"
    if needs_publishing:
        return "Poetry or uv (both support clean packaging/publishing workflows)"
    if team_size > 1:
        return "uv (fastest installs, great for CI/CD and team reproducibility) or Poetry (mature, widely adopted)"
    return "uv (fast, modern default for most new projects in 2025+)"

scenarios = [
    ("script", 1, False),
    ("library", 5, True),
    ("webapp", 8, False),
    ("notebook", 1, False),
]
for project_type, team_size, publish in scenarios:
    print(f"{project_type} (team={team_size}, publish={publish}): {recommend_tool(project_type, team_size, publish)}")